# 066 — Bidirectional RNNs

Every architecture so far reads left to right, so the state at position $t$
knows nothing about $t+1$. When the disambiguating word arrives **after** the
token being labelled, that is fatal. Measured here: forward-only sits at
**chance (49.6%)** and adding the backward pass reaches **63.8%** — and in the
control, where the cue arrives first, the backward pass adds **nothing**.

Untrained random reservoirs with a linear readout, so the absolute numbers are
low on purpose; the comparison is the result.

| Part | What we check |
|---|---|
| A | the task: a cue that arrives after the target |
| B | forward, backward and both |
| C | the control — cue before the target |
| D | what it costs, in parameters and in latency |

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — A task where the answer is downstream

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression

VOCAB, T, TARGET, H = 14, 10, 2, 64

def make(n, seed, cue_before=False):
    r = np.random.default_rng(seed)
    X = r.integers(3, VOCAB, (n, T))
    X[:, TARGET] = 0                                  # the ambiguous token
    y = r.integers(0, 2, n)
    cue = r.integers(0, TARGET, n) if cue_before else r.integers(TARGET + 2, T, n)
    X[np.arange(n), cue] = np.where(y == 1, 1, 2)     # token 1 or 2 is the cue
    return X, y

X, y = make(5, 0)
for row, label in zip(X, y):
    print(f"  {row}  label {label}   (token 0 at position {TARGET} is ambiguous)")
assert (X[:, TARGET] == 0).all()
print()
print("The label is read from the state AT position 2, which is what makes this")
print("per-token rather than sequence classification. Reading the FINAL state")
print("would prove nothing - a forward pass has seen the cue by then either way.")

## Part B — Forward, backward, both

In [ ]:
g = np.random.default_rng(3)
E = g.normal(0, 1, (VOCAB, 16))
Wxf, Whf = g.normal(0, 0.4, (16, H)), g.normal(0, 1, (H, H))
Wxb, Whb = g.normal(0, 0.4, (16, H)), g.normal(0, 1, (H, H))
Whf *= 0.95 / max(abs(np.linalg.eigvals(Whf)))
Whb *= 0.95 / max(abs(np.linalg.eigvals(Whb)))

def states_at(X, pos):
    """Forward and backward hidden states AT pos - not at the sequence end."""
    forward, backward = np.zeros((len(X), H)), np.zeros((len(X), H))
    for i, seq in enumerate(X):
        h = np.zeros(H)
        for t in range(pos + 1):
            h = np.tanh(E[seq[t]] @ Wxf + h @ Whf)
        forward[i] = h
        h = np.zeros(H)
        for t in range(T - 1, pos - 1, -1):
            h = np.tanh(E[seq[t]] @ Wxb + h @ Whb)
        backward[i] = h
    return forward, backward

def run(cue_before, label):
    X_tr, y_tr = make(6000, 0, cue_before)
    X_te, y_te = make(2000, 1, cue_before)
    F_tr, B_tr = states_at(X_tr, TARGET)
    F_te, B_te = states_at(X_te, TARGET)
    print(f"  {label}")
    scores = {}
    for name, tr, te in (("forward only", F_tr, F_te),
                         ("backward only", B_tr, B_te),
                         ("bidirectional", np.hstack([F_tr, B_tr]),
                          np.hstack([F_te, B_te]))):
        scores[name] = LogisticRegression(max_iter=3000).fit(tr, y_tr).score(te, y_te)
        print(f"    {name:<16}{scores[name]:>7.1%}   ({tr.shape[1]} features)")
    return scores

after = run(False, "cue arrives AFTER the token being labelled")
assert abs(after["forward only"] - 0.5) < 0.03          # chance
assert after["bidirectional"] - after["forward only"] > 0.1
print(f"\n  bidirectional gains {after['bidirectional'] - after['forward only']:+.1%}")
print("  and sits marginally BELOW backward-only: on this task the forward half")
print("  carries no signal, only extra parameters to fit.")

## Part C — The control

In [ ]:
before = run(True, "control - cue arrives BEFORE the token being labelled")
print(f"\n  difference: {before['bidirectional'] - before['forward only']:+.1%}")
assert before["forward only"] > 0.95
assert abs(before["bidirectional"] - before["forward only"]) < 0.02
print()
print("Forward alone is already perfect, and reading backwards adds nothing.")
print("Bidirectionality is not free accuracy: it pays exactly when the")
print("information is genuinely downstream of the thing being labelled.")

## Part D — The two costs

In [ ]:
for f_dim, h in ((32, 5), (100, 128)):
    base = f_dim * h + h * h + h
    print(f"  F={f_dim:<4} H={h:<4} RNN {base:>8,}   Bidirectional {2 * base:>9,}")
    assert 2 * base == base * 2
print("  exactly 2x - two independent cells, concatenated")

print("\n  and the cost that is not parameters:")
for use_case, ok in (("offline tagging of complete sentences", True),
                     ("named-entity recognition on stored text", True),
                     ("live transcription as words arrive", False),
                     ("next-word prediction", False)):
    print(f"    {use_case:<42}{'fine' if ok else 'IMPOSSIBLE'}")
print("\n  The backward pass needs the whole sequence before it can start. For")
print("  streaming input that is a latency wall, not merely more compute - and")
print("  for prediction it is cheating, because the answer is in the input.")

## What this notebook established

- With the cue **after** the target: forward-only **49.6%** (chance), backward
  **64.4%**, bidirectional **63.8%** — the backward pass is carrying all of it.
- With the cue **before** the target: forward-only **100%** and bidirectional
  **100%** — the backward pass adds nothing. The control is the result.
- Bidirectional costs **exactly 2x** the parameters, and it cannot be used on
  streaming input or for prediction at all.
- Absolute accuracies are low because these are untrained reservoirs with a
  linear readout; the comparison isolates the architecture question from
  training dynamics.

## Exercises

1. Move the cue to the position immediately after the target. Does forward-only
   recover?
2. Train the readout on the final state instead of the state at position 2 and
   explain why the difference disappears.
3. Replace the backward cell with a copy of the forward cell's weights. How much
   of the gain survives?